# Open mapping theorem: a neighborhood-basis visualization

This notebook is self-contained: it does not need the separate Python script.
Run the cells in order, then use the step, view, test-neighborhood, and zoom controls.

We use $X=Y=\mathbb R^2$, $T(z)=Az$, and the **closed** balls
$$
 U_k=\overline B(0,\rho_k),\qquad \rho_k=0.9\,2^{-k}.
$$
In particular $U_0\subset B(0,1)$, and $U_{k+1}+U_{k+1}=U_k$.
The four views show a single construction from four perspectives.

**Finite-dimensional limitation.** Here the matrix is invertible and images of
closed balls are closed ellipses, so
$F_k=\overline{T(U_k)}=T(U_k)$. Thus the pictures satisfy actual image membership,
which is stronger than the closure membership available in the general proof.
They illustrate its geometry, rather than establish its general closure-removal step.

**Construction, not an inverse solver.** A known limit $x$ is chosen to generate
an admissible spiraling path. The formulas define the path for all $k\geq0$;
only a finite initial segment is displayed.

In [ ]:
# Uncomment if these packages are not installed:
# %pip install numpy matplotlib ipywidgets

In [48]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.axes import Axes

## 1. Construct a valid correction sequence

To generate a visible, spiraling sequence, let $v_k$ be rotating unit vectors and define
$$
 e_k=0.6\rho_{k+1}v_k,\qquad x=e_0,\qquad
 s_k=x-e_k,\qquad y=Tx.
$$
Then $s_0=0$. The next correction satisfies
$$
 x_{k+1}:=s_{k+1}-s_k=e_k-e_{k+1},\qquad
 \|x_{k+1}\|\leq0.9\rho_{k+1}<\rho_{k+1}.
$$
Also $r_k=y-Ts_k=Te_k\in F_{k+1}$ because $e_k\in U_{k+1}$.
These estimates hold at every step. The code additionally checks the sampled
correction bounds, residual identities, nesting inequalities, and Cauchy-tail bounds.

In [ ]:
# ---------- 1. Mathematical construction ----------

def _default_A():
  return [[1.4, 0.6], [-0.25, 0.9]]

  def make_demo(
  steps: int = 8, 
  radius: float = 0.9, 
  angle_degrees: float = 110.0, 
  matrix=None,
  *,
  max_steps: int = 24
) -> dict:
  """Construct a valid infinite spiral, sampled through `steps + 1`.

  U_k = closed_ball(0, rho_k), rho_k = radius * 2^(-k).
  e_k = 0.6 * rho_(k+1) * v_k, with rotating unit vectors v_k.
  x_star = e_0; s_k = x_star - e_k; y = A @ x_star.

  Thus s_0 = 0 and the correction d_(k+1) = s_(k+1) - s_k obeys
      ||d_(k+1)|| <= ||e_k|| + ||e_(k+1)||
                      = 0.9 * rho_(k+1) < rho_(k+1).
  Also r_k = y - A @ s_k = A @ e_k lies in F_(k+1) = A(U_(k+1)).
  These estimates hold for every k, not just the displayed samples.
  """
  if not isinstance(steps, int) or not 1 <= steps <= max_steps:
    raise ValueError("steps must be an integer between 1 and 24.")
  if not np.isfinite(radius) or radius <= 0:
    raise ValueError("radius must be positive and finite.")
  if not np.isfinite(angle_degrees):
    raise ValueError("angle_degrees must be finite.")
  A = np.array(_default_A() if matrix is None else matrix, dtype=float)
  if A.shape != (2, 2) or not np.all(np.isfinite(A)):
    raise ValueError("matrix must be a finite 2-by-2 array.")
  
  singular_values = np.linalg.svd(A, compute_uv=False)
  if singular_values[-1] <= 1e-10 * singular_values[0]:
    raise ValueError("Use a numerically nonsingular matrix.")

  j = np.arange(steps + 3)
  rho_k = radius * 2.0 ** (-j)
  angle = np.deg2rad(30.0 + angle_degrees * j[:-1])
  e_k = 0.6 * rho_k[1:, None] * np.column_stack((np.cos(angle), np.sin(angle)))
  x_star = e_k[0].copy()
  s = x_star - e_k
  y = A @ x_star
  Ts = s @ A.T
  residuals = e_k @ A.T  # More stable than subtracting nearly equal vectors.
  corrections = np.diff(s, axis=0)
  tol = 100 * np.finfo(float).eps * max(1.0, radius)

  cor_norms = np.linalg.norm(corrections, axis=1)

  # Finite-sample checks supplement the analytic estimates in the docstring.
  assert np.allclose(s[0], 0.0)
  assert np.allclose(Ts + residuals, y)
  assert np.all(cor_norms <= rho_k[1:-1] + tol)
  assert np.all(np.linalg.norm(e_k, axis=1) <= rho_k[1:] + tol)
  assert np.all(cor_norms + rho_k[1:-1] <= rho_k[:-2] + tol)
  for n in range(steps + 1):
    assert np.all(np.linalg.norm(s[n:] - s[n], axis=1) <= rho_k[n] + tol)

  return dict(steps=steps, A=A, rho=rho_k, e=e_k, x_star=x_star,
              s=s, y=y, Ts=Ts, residuals=residuals,
              corrections=corrections, opnorm=singular_values[0])

## 2. Draw the four views

**Neighborhood basis:** the irregular *open* set $W$ contains $B(0,\epsilon)$.
Once $\rho_k<\epsilon$, we have $U_k\subset B(0,\epsilon)\subset W$.
The marked finite tails $s_m-s_k$ remain in $U_k$: this is the uniform-in-$m$
control that makes $(s_k)$ Cauchy. The test $\rho_k<\epsilon$ is sufficient;
failure of that test does not itself prove $U_k\not\subset W$.

**Translated neighborhoods:** $C_k=s_k+U_k$.
Since $x_{k+1}\in U_{k+1}$,
$$
 C_{k+1}\subseteq s_k+U_{k+1}+U_{k+1}=C_k.
$$
They are closed, nested, and their diameters tend to zero.

**Images:** $T(C_k)=Ts_k+T(U_k)$ are nested ellipses retaining $y$.

**Residual neighborhoods:** $r_k\in F_{k+1}$ and
$\|r_k\|\leq\|T\|\rho_{k+1}\to0$, so $Ts_k\to y$.

Local zoom changes the coordinate window at every step. Disable it when comparing
absolute sizes. The unfilled outlines depict whole balls or ellipses, not just their boundaries.

In [49]:
# ---------- 2. Drawing helpers and the four proof views ----------
VIEWS = {
  "basis": "1. Neighborhood basis at zero",
  "nested": "2. Nested neighborhoods around partial sums",
  "images": "3. Nested images retaining the target",
  "residuals": "4. Residuals in shrinking image neighborhoods",
}


def _boundary(center, radius, A=None):
  theta = np.linspace(0, 2 * np.pi, 361)
  p = radius * np.column_stack((np.cos(theta), np.sin(theta)))
  if A is not None:
      p = p @ np.asarray(A).T
  return np.asarray(center) + p


def _curve(ax: Axes, points, **kwargs):
  return ax.plot(points[:, 0], points[:, 1], **kwargs)[0]

def _point(ax: Axes, point, **kwargs):
  return ax.plot(*point, **kwargs)[0]

def _new_figure(title):
  fig = plt.figure(figsize=(8.2, 7.4))
  ax = fig.add_axes([0.13, 0.31, 0.76, 0.57], axes_class=Axes)
  ax.set_aspect("equal", adjustable="box")
  ax.set(xlabel="First coordinate", ylabel="Second coordinate")
  ax.grid(True, alpha=0.18)
  fig.suptitle(title, fontsize=14, y=0.96)
  return fig, ax


def _limits(ax: Axes, center, half_width):
  center = np.asarray(center)
  ax.set(
    xlim=(center[0] - half_width, center[0] + half_width),
    ylim=(center[1] - half_width, center[1] + half_width),
  )


def _finish(fig, ax: Axes, text):
  handles, labels = ax.get_legend_handles_labels()
  fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.245),
              ncol=3, frameon=False, fontsize=9)
  fig.text(0.06, 0.025, text, fontsize=10, va="bottom", linespacing=1.7)
  return fig

def _w_radius(eps, theta):
  return eps * (1.45 + 0.25 * np.cos(3 * theta) + 0.15 * np.sin(5 * theta))
def plot_view(
  demo: dict, 
  view: str = "nested", 
  k: int = 2,
  epsilon: float = 0.25, 
  zoom: bool = False
):
  """Return one Matplotlib figure; epsilon affects the basis view only.

  Dashed outlines denote either the next ball/image or the test
  neighborhood, as labeled. The unfilled outlines represent whole
  closed balls or ellipses, not just their boundaries.
  """
  nsteps = demo['steps']
  if view not in VIEWS:
    raise ValueError(f"view must be one of {tuple(VIEWS)}")
  if not isinstance(k, (int, np.integer)) or not 0 <= k <= nsteps:
    raise ValueError(f"k must be between 0 and {nsteps}.")
  if not np.isfinite(epsilon) or epsilon <= 0:
    raise ValueError("epsilon must be positive and finite.")
  A, rho, s = demo["A"], demo["rho"], demo["s"]
  x_star, y, Ts = demo["x_star"], demo["y"], demo["Ts"]
  residuals, opnorm = demo["residuals"], demo["opnorm"]
  fig, ax = _new_figure(VIEWS[view] + f"  |  k = {k}")
  zero = np.zeros(2)

  cmap = plt.get_cmap('Greens')
  cmap2 = plt.get_cmap('Reds')

  def _plot_boundary(ax, points, j, label):
    kwargs = dict(
      linewidth = 2.5 if j == k else 1.2,
      linestyle = "--" if j == k + 1 else "-",
      c = (cmap(j/nsteps) if j>= k else cmap2(j/nsteps)),
      alpha = 1.0 if j >= k else 0.3,
    )
    return _curve(ax, points, label=label, **kwargs)

  if view == "basis":
    for j in range(k + 2):
      label = (rf"$U_{j}$" if j in {k, k + 1} else "_nolegend_")
      _plot_boundary(ax, _boundary(zero, rho[j]), j, label)

    theta = np.linspace(0, 2 * np.pi, 601)
    # W is the OPEN interior of this radial boundary. Its radius is
    # >= 1.05 * epsilon, so B(0, epsilon) is certainly contained in W.
    w = _w_radius(epsilon, theta)[:, None] * np.column_stack((np.cos(theta), np.sin(theta)))
    _curve(ax, w, linestyle="--", linewidth=1.8, c='slategray', label=r"Boundary of $W$")
    _curve(ax, _boundary(zero, epsilon), linestyle=":", linewidth=1.5, c='mediumpurple',
            label=r"Boundary of $B(0,\epsilon)$")
    _point(ax, zero, marker="+", markersize=10, linestyle="None")
    # Some finite tails illustrate the uniform-in-m Cauchy estimate.
    tails = s[k + 1:] - s[k]
    _curve(ax, tails, marker="o", markersize=3, linewidth=0.9, c='black',
            label=r"Tails $s_m-s_k$, $m>k$")
    _limits(ax, zero, 1.22 * rho[k] if zoom
            else 1.12 * max(rho[0], 1.85 * epsilon))
    status = (
      rf"$\rho_{k}<\epsilon$: $U_{k}\subset B(0,\epsilon)\subset W$."
      if rho[k] < epsilon else
      r"The chosen $\epsilon$ does not yet certify $U_k\subset W$."
    )
    text = (
      rf"$\rho_k={rho[k]:.5g}$; $\epsilon={epsilon:.5g}$.  " + status
      + "\n" + r"$U_{k+1}+U_{k+1}=U_k$; consequently "
      + r"$s_m-s_k\in U_k$ for every $m>k$."
      + "\n" + r"Shrinking $U_k$ eventually fit inside any prescribed neighborhood of $0$."
    )

  elif view == "nested":
    for j in range(k + 2):
      label = rf"$C_{j}=s_{j}+U_{j}$" if j in {k, k + 1} else "_nolegend_"
      _plot_boundary(ax, _boundary(s[j], rho[j]), j, label)

    _curve(ax, s[:k + 2], marker="o", markersize=3.5, linewidth=1.0, label="Partial sums")
    _point(ax, s[k], marker="s", markersize=7, linestyle="None", label=rf"$s_{k}$")
    _point(ax, x_star, marker="*", markersize=13, linestyle="None", label=r"Limit $x$")
    ax.annotate("", xy=s[k + 1], xytext=s[k], arrowprops=dict(arrowstyle="->", linewidth=1.3))
    _limits(ax, s[k] if zoom else zero, 1.22 * (rho[k] if zoom else rho[0]))
    move = np.linalg.norm(s[k + 1] - s[k])
    text = (
      rf"$\|s_{{k+1}}-s_k\|+\rho_{{k+1}}="
      rf"{move + rho[k + 1]:.5g}\leq {rho[k]:.5g}=\rho_k$." + "\n" 
      + r"$C_{k+1}=s_k+x_{k+1}+U_{k+1}\subseteq s_k+U_k=C_k$." + "\n" 
      + r"Closed, nested sets; $\mathrm{diam}(C_k)=2\rho_k\to0$; "
      + r"$\bigcap_{k\geq0}C_k=\{x\}$."
    )

  elif view == "images":
    for j in range(k + 2):
      label = rf"$T(C_{j})$" if j in {k, k + 1} else "_nolegend_"
      _plot_boundary(ax, _boundary(Ts[j], rho[j], A), j, label)

    _curve(ax, Ts[:k + 2], marker="o", markersize=3.5, linewidth=1.0, label=r"Images $Ts_j$")
    _point(ax, Ts[k], marker="s", markersize=7, linestyle="None", label=rf"$Ts_{k}$")
    _point(ax, y, marker="*", markersize=13, linestyle="None", label=r"Target $y$")
    ax.annotate("", xy=y, xytext=Ts[k],
                arrowprops=dict(arrowstyle="->", linewidth=1.3))
    _limits(ax, Ts[k] if zoom else zero,
            1.22 * opnorm * (rho[k] if zoom else rho[0]))
    text = (
      r"The arrow from $Ts_k$ to $y$ is the residual $r_k=y-Ts_k$."
      + "\n" + r"$y\in T(C_{k+1})\subseteq T(C_k)$ in this example; "
      + r"$\|r_k\|=" + f"{np.linalg.norm(residuals[k]):.5g}" + r"$."
      + "\n" + r"As $C_k$ shrink to $x$, their images shrink to $Tx=y$."
    )

  else:  # residuals
    for j in range(1, k + 2):
      label = rf"$F_{j}=T(U_{j})$" if j in {max(1, k), k + 1} else "_nolegend_",
      _plot_boundary(ax, _boundary(zero, rho[j], A), j, label)

    _curve(ax, residuals[:k + 1], marker="o", markersize=3.5,
            linewidth=1.0, label=r"Residuals $r_0,\ldots,r_k$")
    _point(ax, residuals[k], marker="s", markersize=7, linestyle="None", label=rf"$r_{k}$")
    _point(ax, zero, marker="+", markersize=11, linestyle="None", label=r"Limit $0$")
    _limits(ax, zero, 1.22 * opnorm * (rho[k + 1] if zoom else rho[1]))
    text = (
      r"$r_k=T(e_k)$ and $\|e_k\|=0.6\rho_{k+1}<\rho_{k+1}$; hence $r_k\in F_{k+1}$."
      + "\n" + r"$\|r_k\|\leq\|T\|\rho_{k+1}="
      + f"{opnorm * rho[k + 1]:.5g}" + r"\longrightarrow0$."
      + "\n" + r"Continuity and $s_k\to x$ give $Ts_k\to Tx$; "
      + r"also $Ts_k=y-r_k\to y$."
    )

  ax.ticklabel_format(style="sci", axis="both", scilimits=(-3, 3), useOffset=False)
  return _finish(fig, ax, text)



## 3. Interactive controls

The notebook uses `ipywidgets.interactive_output` and inline Matplotlib; it does not require `ipympl`. The epsilon control only affects the neighborhood-basis view.

In [50]:
# ---------- 3. Optional notebook controls ----------
def notebook_app(demo=None):
    """Display live controls. Uses inline Matplotlib, so no ipympl is needed."""
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError as exc:
        raise ImportError("Install ipywidgets, or use the exported HTML explorer.") from exc
    if demo is None:
        demo = make_demo()

    step = widgets.IntSlider(value=min(2, demo["steps"]), min=0, max=demo["steps"],
                             description="Step k", continuous_update=False)
    eps = widgets.FloatLogSlider(value=0.25, base=10, min=-3, max=0.2,
                                 step=0.05, description="epsilon",
                                 readout_format=".3g", continuous_update=False)
    zoom = widgets.Checkbox(value=False, description="Local zoom")
    view = widgets.Dropdown(options=[(title, key) for key, title in VIEWS.items()]
                            + [("Show all four separately", "all")],
                            value="nested", description="View")

    def update(k, epsilon, zoom, view):
        for name in (VIEWS if view == "all" else [view]):
            fig = plot_view(demo, name, k, epsilon, zoom)
            display(fig)
            plt.close(fig)

    output = widgets.interactive_output(update, dict(k=step, epsilon=eps,
                                                      zoom=zoom, view=view))
    ui = widgets.VBox([view, step, eps, zoom, output])
    display(ui)
    return ui




In [ ]:
demo = make_demo(steps=24, radius=0.9, angle_degrees=10.0)
ui = notebook_app(demo)

## 4. Static use, without widgets

This cell works with NumPy and Matplotlib alone. Change `view` to `basis`, `nested`, `images`, or `residuals`.

In [ ]:
demo = make_demo(steps=10)
fig = plot_view(demo, view="nested", k=3, epsilon=0.1, zoom=True)
plt.show()
# fig.savefig("nested_neighborhoods.png", dpi=180, bbox_inches="tight")

## A useful experiment

Select **Neighborhood basis**, turn off zoom, and choose a small $\epsilon$.
Increase $k$ until $U_k$ fits inside $B(0,\epsilon)$. Decrease $\epsilon$ and
repeat: the same sequence eventually enters the smaller prescribed neighborhood.

Then select **Nested neighborhoods** and enable local zoom. Watch the next
center move while its *entire next ball* remains inside the current ball.
The inequality below the plot certifies that containment. Finally inspect the
image and residual views at the same value of $k$.

The general proof uses completeness to obtain the common point and continuity
to identify its image with the retained target. In this model those conclusions
can also be read directly from the explicit formulas.